# EPCLM Core-001 Topology-001

Tesla T4 / FP32；R1、IC、R2、SR、T4 顺序运行，各 250K Prediction Tokens。共享同一份初始参数，保持广播事件注入、K=4、Alpha=0.1、16×128 状态。
启用 Internet、GITHUB_TOKEN 与 KAGGLE_API_TOKEN Secrets，Save & Run All。无需挂载历史 Output。
全新训练，不下载旧权重、不创建周期 Checkpoint 或 Recovery Bundle。已完成组只复用报告，不重复训练。中断组从头运行。
训练后执行完整 Validation/Holdout、配对 Bootstrap、状态传播、状态多样性、成本/推理测量和辅助生成。失败组或失败诊断不阻断其他结果发布。
预算 5400 秒执行 + 600 秒发布 + 1200 秒余量；时间不足标记 INCOMPLETE，不自动扩展。最终模型只保存在各组记录的 Kaggle Output，不进入 Git。
单 Seed Development Screen；不自动进入 Phase 2 或多 Seed 训练。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-topology-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-topology-001'
OUTPUT = WORKSPACE / 'epclm-topology-001-bootstrap'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()
if not os.environ['KAGGLE_API_TOKEN']:
    raise RuntimeError('Enable the existing Kaggle Secret KAGGLE_API_TOKEN.')

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; recoverable output, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('epclm-topology-001-' + SOURCE_COMMIT[:12])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before running T1 formal validation/training.')
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/epclm-core-001/topology-001/experiment.py'),
    '--output', str(OUTPUT)], cwd=REPO, timeout_seconds=6000)
